# VLM-Anomaly — Claude Opus Few-Shot Ensemble (fewshot2-ens4)

**Model:** `claude-opus-4-7`  
**Strategy:** 2 normal reference images cached in system message + 4 ensemble prompts per category  
**Scope:** 8 underperforming categories: capsule, screw, zipper, carpet, pill, transistor, leather, grid  
**Expected AUROC improvement:** From ~0.60–0.68 range → target >0.75  

API design:
- Reference images sent once per batch, cached with `cache_control: ephemeral` (0.1× token cost)
- 10 test images per batch per prompt = 4 API calls per 10 images per category
- Final score = mean confidence across 4 prompts (expert, compare, cot, negative)
- Model ID in results: `anthropic/claude-opus-4-7-fewshot2-ens4`

In [4]:
# ── Cell 1: Setup paths & sys.path ──────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path().resolve().parent
SRC_DIR   = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

PROMPTS_DIR  = REPO_ROOT / 'prompts'
ENHANCED_YAML = PROMPTS_DIR / 'claude_opus_enhanced.yaml'
RESULTS_DIR  = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root    : {REPO_ROOT}')
print(f'Enhanced YAML: {ENHANCED_YAML}  (exists={ENHANCED_YAML.exists()})')
print(f'Results dir  : {RESULTS_DIR}')

Repo root    : /Users/sabareeswarans/Projects_26/VLM-Anomaly
Enhanced YAML: /Users/sabareeswarans/Projects_26/VLM-Anomaly/prompts/claude_opus_enhanced.yaml  (exists=True)
Results dir  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [7]:
# ── Cell 2: Verify API key ───────────────────────────────────────────────────
import os

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert api_key, (
    'ANTHROPIC_API_KEY not set — add it to your .env file or export it:\n'
    '  export ANTHROPIC_API_KEY=sk-ant-...'
)
print(f'API key : sk-ant-...{api_key[-6:]}  (length={len(api_key)})')
print('API key present.')

API key : sk-ant-...3uSAAA  (length=108)
API key present.


In [8]:
# ── Cell 3: Find MVTec dataset ───────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
    Path('/kaggle/input/mvtec-ad'),
]:
    if (candidate / 'bottle').exists():
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, (
    'MVTec not found. Run: bash scripts/download_mvtec.sh\n'
    f'(searched: {REPO_ROOT}/data/mvtec, /tmp/mvtec)'
)

print(f'MVTec root: {MVTEC_ROOT}')
cats = sorted(d.name for d in MVTEC_ROOT.iterdir() if d.is_dir())
print(f'Categories: {cats}')

MVTec root: /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [9]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL = 'claude-opus-4-7'
LIMIT = None    # None = all test images; set e.g. 20 for a quick smoke run

# These 8 categories had the lowest AUROC in the vanilla Opus baseline
TARGET_CATEGORIES = [
    'capsule',     # baseline AUROC 0.671
    'screw',       # baseline AUROC 0.614
    'zipper',      # baseline AUROC 0.692
    'carpet',      # baseline AUROC 0.651
    'pill',        # baseline AUROC 0.688
    'transistor',  # baseline AUROC 0.611
    'leather',     # baseline AUROC 0.700
    'grid',        # baseline AUROC 0.682
]

print(f'Model      : {MODEL}')
print(f'Categories : {TARGET_CATEGORIES}')
print(f'Limit      : {LIMIT or "all"} images per category')

Model      : claude-opus-4-7
Categories : ['capsule', 'screw', 'zipper', 'carpet', 'pill', 'transistor', 'leather', 'grid']
Limit      : all images per category


In [10]:
# ── Cell 5: Build shared objects ─────────────────────────────────────────────
from vlm_anomaly.backends.anthropic_backend import AnthropicBackend
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.evaluators.few_shot_evaluator import FewShotEnsembleEvaluator
from vlm_anomaly.logging import configure_logging

configure_logging(log_level='INFO')

backend = AnthropicBackend(model=MODEL)

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
)
settings.results_dir = RESULTS_DIR

dataset = MVTec(root_dir=MVTEC_ROOT)

evaluator = FewShotEnsembleEvaluator(
    backend=backend,
    dataset=dataset,
    categories=TARGET_CATEGORIES,
    enhanced_yaml=ENHANCED_YAML,
    limit=LIMIT,
    settings=settings,
)

print(f'Backend  : {backend.name} / {MODEL}')
print(f'Dataset  : {MVTEC_ROOT}')
print(f'YAML     : {ENHANCED_YAML}')
print('Ready — no budget cap, runs to completion.')

Backend  : anthropic / claude-opus-4-7
Dataset  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
YAML     : /Users/sabareeswarans/Projects_26/VLM-Anomaly/prompts/claude_opus_enhanced.yaml
Ready — no budget cap, runs to completion.


In [6]:
# ── Cell 6: Smoke test — verify reference injection before full run ──────────
# Tests 1 batch (2 images) × 1 prompt for capsule.
import yaml

with open(ENHANCED_YAML) as f:
    cfg = yaml.safe_load(f)

cat_cfg = cfg['categories']['capsule']
system_text = cat_cfg['system'].strip()
prompt_text = cat_cfg['prompts'][0]['text'].strip()   # expert prompt

# Pick 2 reference images from train/good
ref_paths = sorted((MVTEC_ROOT / 'capsule' / 'train' / 'good').glob('*.png'))[:2]
# Pick 2 test images (1 good, 1 defect)
smoke_imgs = [
    next((MVTEC_ROOT / 'capsule' / 'test' / 'good').glob('*.png')),
    next((MVTEC_ROOT / 'capsule' / 'test' / 'crack').glob('*.png')),
]

print(f'Ref images  : {[r.name for r in ref_paths]}')
print(f'Test images : {[i.name for i in smoke_imgs]}')
print()

smoke_preds = backend.predict_batch_with_refs(
    ref_images=ref_paths,
    images=smoke_imgs,
    system_text=system_text,
    prompt=prompt_text,
)

for img, pred in zip(smoke_imgs, smoke_preds):
    print(f'  {img.parent.name}/{img.name}')
    print(f'    is_anomalous : {pred.is_anomalous}')
    print(f'    confidence   : {pred.confidence:.2f}')
    print(f'    defect_type  : {pred.defect_type}')
    print(f'    tokens_in    : {pred.tokens_in}  (cache_write logged in structlog)')
    print(f'    cost_usd     : ${pred.cost_usd:.6f}')
    print(f'    parse_error  : {pred.parse_error}')
    print()

print('Smoke test passed — reference injection and prompt caching are working.')

Ref images  : ['000.png', '001.png']
Test images : ['002.png', '002.png']



2026-05-24T23:55:06.003993Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-24T23:55:06.007512Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=2 cost_usd=0.10799 latency_ms=5258 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=0 tokens_cache_w=2722 tokens_in=2867 tokens_out=186


  good/002.png
    is_anomalous : False
    confidence   : 0.85
    defect_type  : None
    tokens_in    : 1433  (cache_write logged in structlog)
    cost_usd     : $0.053996
    parse_error  : False

  crack/002.png
    is_anomalous : True
    confidence   : 0.80
    defect_type  : mechanical deformation
    tokens_in    : 1433  (cache_write logged in structlog)
    cost_usd     : $0.053996
    parse_error  : False

Smoke test passed — reference injection and prompt caching are working.


In [6]:
# ── Cell 7: Run few-shot ensemble on 8 target categories ─────────────────────
# Idempotent — checks for existing JSONL before running any category.
# Each batch = 10 test images × 4 prompts = 4 API calls per batch.
import json
from sklearn.metrics import roc_auc_score

MODEL_ID = 'anthropic/claude-opus-4-7-fewshot2-ens4'


def _load_existing_fewshot(results_dir, category):
    """Return rows from the most recent completed run, or []."""
    for f in sorted(results_dir.glob(f'*_mvtec_{category}_fewshot_ens.jsonl'),
                    key=lambda p: p.stat().st_mtime, reverse=True):
        if f.stat().st_size < 10:
            continue
        try:
            lines = [l for l in f.read_text().splitlines() if l.strip()]
            if lines and json.loads(lines[0]).get('model_id') == MODEL_ID:
                return lines
        except Exception:
            pass
    return []


# Separate categories into already-done vs still-needed
all_results = []
todo = []
for category in TARGET_CATEGORIES:
    existing = _load_existing_fewshot(RESULTS_DIR, category)
    if existing:
        rows = [json.loads(l) for l in existing]
        labels = [r['sample_label'] for r in rows]
        scores  = [r['prediction']['confidence'] for r in rows]
        try:
            auroc = roc_auc_score(labels, scores) if len(set(labels)) > 1 else float('nan')
        except Exception:
            auroc = float('nan')
        print(f'  [skip] {category:12s}  {len(rows)} rows  AUROC={auroc:.3f}')
        all_results.append({'category': category, 'auroc': auroc, 'n': len(rows), 'skipped': True})
    else:
        todo.append(category)

if todo:
    print(f'\nRunning {len(todo)} remaining: {todo}')
    from vlm_anomaly.evaluators.few_shot_evaluator import FewShotEnsembleEvaluator
    run_evaluator = FewShotEnsembleEvaluator(
        backend=backend,
        dataset=dataset,
        categories=todo,   # only the ones not yet done
        enhanced_yaml=ENHANCED_YAML,
        limit=LIMIT,
        settings=settings,
    )
    # Single call — internally batches 10 imgs/call × 4 prompts per category
    for er in run_evaluator.run():
        print(f'  {er.category:12s}  AUROC={er.auroc:.3f}  F1={er.f1:.3f}  ${er.total_cost_usd:.3f}')
        all_results.append({
            'category': er.category, 'auroc': er.auroc,
            'f1': er.f1, 'n': er.n_images,
            'cost': er.total_cost_usd, 'skipped': False,
        })
else:
    print('All categories already done.')

print(f'\nTotal: {len(all_results)} categories.')

2026-05-25T00:24:35.906684Z [info     ] few_shot_eval.run.start        [vlm_anomaly.evaluators.few_shot_evaluator] budget_usd=None categories=['screw', 'zipper', 'carpet', 'pill', 'transistor', 'leather', 'grid'] experiment_id=870f4a02 model_id=anthropic/claude-opus-4-7-fewshot2-ens4


  [skip] capsule       120 rows  AUROC=0.684

Running 7 remaining: ['screw', 'zipper', 'carpet', 'pill', 'transistor', 'leather', 'grid']
  [screw] starting — 160 images, batch_size=20


2026-05-25T00:24:53.412974Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:24:53.417046Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.55609 latency_ms=17490 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=0 tokens_cache_w=2835 tokens_in=27784 tokens_out=1149
2026-05-25T00:25:16.666465Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:25:16.670319Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.50869 latency_ms=23246 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2835 tokens_cache_w=0 tokens_in=27784 tokens_out=1169
2026-05-25T00:25:36.278490Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:25:36.282390Z [info     ] anthropic.predict_batch_with_refs [

    screw  20/160 images done  (batch 1/8)


2026-05-25T00:26:12.023014Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:26:12.027248Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.50299 latency_ms=16624 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2835 tokens_cache_w=0 tokens_in=27784 tokens_out=1093
2026-05-25T00:26:30.320126Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:26:30.323504Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.50621 latency_ms=18289 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2835 tokens_cache_w=0 tokens_in=27784 tokens_out=1136
2026-05-25T00:26:49.777565Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:26:49.781051Z [info     ] anthropic.predict_batch_with_refs [

...

    zipper  140/151 images done  (batch 7/8)


2026-05-25T00:44:05.892960Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:44:05.896033Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=11 cost_usd=0.28471 latency_ms=11945 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2839 tokens_cache_w=0 tokens_in=15422 tokens_out=655
2026-05-25T00:44:18.199802Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:44:18.202917Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=11 cost_usd=0.28656 latency_ms=12301 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2839 tokens_cache_w=0 tokens_in=15415 tokens_out=681
2026-05-25T00:44:35.253242Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:44:35.257319Z [info     ] anthropic.predict_batch_with_refs [vl

    zipper  151/151 images done  (batch 8/8)

  [zipper] DONE  n=151  AUROC=0.4150  F1=0.8333  cost=$15.505

  [carpet] starting — 117 images, batch_size=20


2026-05-25T00:45:07.484012Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:45:07.487401Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.55875 latency_ms=19420 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=0 tokens_cache_w=2836 tokens_in=27795 tokens_out=1182
2026-05-25T00:45:27.304104Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:45:27.308057Z [info     ] anthropic.predict_batch_with_refs [vlm_anomaly.backends.anthropic_backend] batch_size=20 cost_usd=0.51124 latency_ms=19810 model=claude-opus-4-7 n_refs=2 parse_errors=0 tokens_cache_r=2836 tokens_cache_w=0 tokens_in=27799 tokens_out=1200
2026-05-25T00:45:47.987806Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK" [httpx]
2026-05-25T00:45:47.991285Z [info     ] anthropic.predict_batch_with_refs [

KeyboardInterrupt: 

In [5]:
# ── Cell 7b: Per-category results table (independent — reads from DuckDB) ────
import importlib, pandas as pd
import vlm_anomaly.analysis.aggregator as _agg_mod
importlib.reload(_agg_mod)
from vlm_anomaly.analysis.aggregator import leaderboard

ENS_MODEL_ID = 'anthropic/claude-opus-4-7-fewshot2-ens4'

lb = leaderboard(RESULTS_DIR)
df_ens = (
    lb[lb['model_id'] == ENS_MODEL_ID]
    [['category', 'auroc', 'f1', 'n_images', 'total_cost_usd']]
    .rename(columns={'n_images': 'n', 'total_cost_usd': 'cost'})
    .sort_values('auroc', ascending=False)
    .reset_index(drop=True)
)

n_cats     = df_ens['auroc'].notna().sum()
mean_auroc = df_ens['auroc'].mean()
print(f'Categories with results : {n_cats}')
print(f'Mean AUROC              : {mean_auroc:.4f}  (based on {n_cats} categories)')
print()
display(df_ens)

Categories with results : 2
Mean AUROC              : 0.6306  (based on 2 categories)



,category,auroc,f1,n,cost
0,capsule,0.683550,0.874317,120.0,12.388979
1,screw,0.577577,0.772093,160.0,16.418314


In [ ]:
# ── Cell 8: Leaderboard — few-shot ens4 vs vanilla Opus baseline ─────────────
import importlib
import pandas as pd
import vlm_anomaly.analysis.aggregator as _agg_mod
importlib.reload(_agg_mod)
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

ENS_MODEL_ID   = 'anthropic/claude-opus-4-7-fewshot2-ens4'
BASELINE_MODEL = 'anthropic/claude-opus-4-7'

lb = leaderboard(RESULTS_DIR)

# ── All 15 baseline categories ────────────────────────────────────────────────
base_all = lb[lb['model_id'] == BASELINE_MODEL][['category', 'auroc']].set_index('category')
ens_cats = lb[lb['model_id'] == ENS_MODEL_ID][['category', 'auroc']].set_index('category')

# Consolidated: start from all 15 baseline AUROCs, replace only the ensemble-run ones
consolidated = base_all['auroc'].copy()
for cat in ens_cats.index:
    if cat in consolidated.index:
        consolidated[cat] = ens_cats.loc[cat, 'auroc']

baseline_mean     = base_all['auroc'].mean()          # mean of all 15
consolidated_mean = consolidated.mean()                # all 15 with replacements
replaced_cats     = sorted(ens_cats.index.tolist())

print(f'Baseline mean AUROC (all 15 cats) : {baseline_mean:.4f}')
print(f'Consolidated mean AUROC           : {consolidated_mean:.4f}')
print(f'  (15 cats — {len(replaced_cats)} replaced by ensemble: {replaced_cats})')
print(f'Delta                             : {consolidated_mean - baseline_mean:+.4f}')
print()

# ── Per-category comparison for the ensemble-run categories only ──────────────
if not ens_cats.empty:
    comp = (
        pd.concat([
            ens_cats.rename(columns={'auroc': ENS_MODEL_ID}),
            base_all.loc[base_all.index.isin(ens_cats.index)].rename(columns={'auroc': BASELINE_MODEL}),
        ], axis=1)
        .round(4)
    )
    comp['delta'] = (comp[ENS_MODEL_ID] - comp[BASELINE_MODEL]).round(4)
    print('=== Per-category: Baseline vs Ensemble ===')
    display(comp)
    print()

# ── Full leaderboard summary ──────────────────────────────────────────────────
summary = cost_accuracy_table(RESULTS_DIR)
print('=== Summary (mean AUROC — ensemble shown as consolidated 15-cat) ===')
# Override the ensemble row with the consolidated mean
summary_display = summary.copy()
mask = summary_display['model_id'] == ENS_MODEL_ID
summary_display.loc[mask, 'mean_auroc'] = consolidated_mean
summary_display = summary_display.sort_values('mean_auroc', ascending=False)
print(summary_display[['model_id', 'mean_auroc', 'mean_latency_ms']].to_string(index=False))

In [12]:
# ── Cell 9: Add ensemble model to REPORT.md leaderboard ─────────────────────
# Logic:
#   - Take the vanilla Opus-4-7 per-category AUROCs for all 15 categories.
#   - Replace capsule + screw (the two ensemble-run cats) with their ensemble AUROCs.
#   - Mean of the resulting 15 values = approximate consolidated ensemble AUROC.
#   - Append a leaderboard table to REPORT.md that includes this row.

import pandas as pd
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table
from vlm_anomaly.analysis.report_generator import generate
import importlib, vlm_anomaly.analysis.aggregator as _agg
importlib.reload(_agg)
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

BASELINE_ID = 'anthropic/claude-opus-4-7'
ENSEMBLE_ID = 'anthropic/claude-opus-4-7-fewshot2-ens4'

lb = leaderboard(RESULTS_DIR)

base_per_cat = lb[lb['model_id'] == BASELINE_ID][['category','auroc']].set_index('category')
ens_per_cat  = lb[lb['model_id'] == ENSEMBLE_ID][['category','auroc']].set_index('category')

# Swap ensemble results into the 15-cat baseline map
consolidated = base_per_cat['auroc'].copy()
for cat in ens_per_cat.index:
    if cat in consolidated.index:
        consolidated[cat] = ens_per_cat.loc[cat, 'auroc']

baseline_mean    = base_per_cat['auroc'].mean()
consolidated_mean = consolidated.mean()
ens_cats         = sorted(ens_per_cat.index.tolist())
n_ens            = len(ens_cats)

print(f'Baseline mean AUROC (15 cats) : {baseline_mean:.4f}')
print(f'Consolidated mean AUROC       : {consolidated_mean:.4f}  '
      f'(baseline + {n_ens} ensemble replacements: {ens_cats})')
print(f'Delta                         : {consolidated_mean - baseline_mean:+.4f}')

# ── Regenerate base report ────────────────────────────────────────────────────
report_path = REPO_ROOT / 'REPORT.md'
generate(RESULTS_DIR, str(report_path))

# ── Build leaderboard table rows ─────────────────────────────────────────────
summary = cost_accuracy_table(RESULTS_DIR)

# Add the ensemble row with consolidated AUROC (replacing its raw partial mean)
ensemble_cost = lb[lb['model_id'] == ENSEMBLE_ID]['total_cost_usd'].sum()
ensemble_lat  = lb[lb['model_id'] == ENSEMBLE_ID]['mean_latency_ms'].mean()

ensemble_row = pd.DataFrame([{
    'model_id'    : ENSEMBLE_ID,
    'mean_auroc'  : consolidated_mean,
    'mean_latency_ms': ensemble_lat,
    'note'        : f'approx — {n_ens}/15 cats replaced',
}])

# Pull other models from summary
other = summary[summary['model_id'] != ENSEMBLE_ID][
    ['model_id', 'mean_auroc', 'mean_latency_ms']
].copy()
other['note'] = ''

leaderboard_df = (
    pd.concat([other, ensemble_row[['model_id','mean_auroc','mean_latency_ms','note']]])
    .sort_values('mean_auroc', ascending=False)
    .reset_index(drop=True)
)

# ── Append leaderboard section to REPORT.md ───────────────────────────────────
lines = [
    '\n---\n\n',
    '## Leaderboard with Few-Shot Ensemble\n\n',
    f'> Ensemble model consolidated AUROC: baseline 15-category AUROCs with '
    f'{n_ens} categories replaced by ensemble results ({", ".join(ens_cats)}).\n\n',
    '| Model | Mean AUROC | Latency (ms) | Note |\n',
    '|-------|-----------|-------------|------|\n',
]
for _, row in leaderboard_df.iterrows():
    auroc = f"{row['mean_auroc']:.4f}" if pd.notna(row['mean_auroc']) else 'n/a'
    lat   = f"{row['mean_latency_ms']:.0f}" if pd.notna(row['mean_latency_ms']) else 'n/a'
    note  = row.get('note', '') or ''
    lines.append(f"| {row['model_id']} | {auroc} | {lat} | {note} |\n")

with open(report_path, 'a') as f:
    f.writelines(lines)

print(f'\nAppended leaderboard to {report_path}')

2026-05-25T01:08:57.013036Z [info     ] report.generate.start          [vlm_anomaly.analysis.report_generator] results_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results


Baseline mean AUROC (15 cats) : 0.7519
Consolidated mean AUROC       : 0.7709  (baseline + 2 ensemble replacements: ['capsule', 'screw'])
Delta                         : +0.0190


2026-05-25T01:09:02.302796Z [info     ] report.generate.done           [vlm_anomaly.analysis.report_generator] plots=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results/plots report=/Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md



Appended leaderboard to /Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md
